# Fine-tune GLM-OCR for Vietnamese Diacritical Marks

Follows the official guide: [examples/finetune/README.md](https://github.com/zai-org/GLM-OCR/blob/main/examples/finetune/README.md)

**Dataset format:** ShareGPT with `messages`/`role`/`content` + `images` fields.

**Requirements:**
- GPU: T4 (16GB) or better
- Upload `vietnamese_ocr.zip` to Google Drive `My Drive` root
  - Structure: `vietnamese_ocr/vietnamese_ocr.json` + `vietnamese_ocr/images/txt_*.png`

## 1. Check GPU

In [ ]:
!nvidia-smi

## 2. Mount Drive & Extract Dataset

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# Extract dataset
!cp "/content/drive/My Drive/vietnamese_ocr.zip" /content/
!cd /content && unzip -q -o vietnamese_ocr.zip
!echo "Images:" $(ls /content/vietnamese_ocr/images/txt_*.png | wc -l)
!echo "JSON:" $(ls -lh /content/vietnamese_ocr/vietnamese_ocr.json)

In [ ]:
# Verify dataset format: must have messages/role/content
import json
with open("/content/vietnamese_ocr/vietnamese_ocr.json", "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"Total samples: {len(data)}")
print(f"\nSample 0:")
print(json.dumps(data[0], ensure_ascii=False, indent=2))

# Verify format
sample = data[0]
assert "messages" in sample, "ERROR: missing 'messages' key"
assert "images" in sample, "ERROR: missing 'images' key"
assert sample["messages"][0]["role"] == "user", "ERROR: first message must be user"
assert sample["messages"][1]["role"] == "assistant", "ERROR: second message must be assistant"
assert "<image>" in sample["messages"][0]["content"], "ERROR: user message must contain <image>"
print("\n✓ Dataset format is correct!")

# Check image exists
import os
img_path = os.path.join("/content/vietnamese_ocr", sample["images"][0])
print(f"\nImage path: {img_path}")
print(f"Image exists: {os.path.exists(img_path)}")

## 3. Install LLaMA-Factory

Following the official GLM-OCR finetune guide.

In [ ]:
!git clone --depth 1 https://github.com/hiyouga/LLaMA-Factory.git
%cd /content/LLaMA-Factory
!pip install -e ".[torch,metrics]" 2>&1 | tail -5

In [ ]:
# Pin transformers to 5.6.0 — compatible with both LLaMA-Factory (>=4.55, <=5.6.0) and GLM-OCR (>=5.3.0)
!pip install transformers==5.6.0 2>&1 | tail -3

In [ ]:
# Verify LLaMA-Factory
!llamafactory-cli version

## 4. Download GLM-OCR Model

In [ ]:
from huggingface_hub import snapshot_download

model_dir = snapshot_download(
    "zai-org/GLM-OCR",
    local_dir="/content/GLM-OCR",
    local_dir_use_symlinks=False,
)
print(f"Model downloaded to: {model_dir}")

## 5. Prepare Dataset for LLaMA-Factory

Copy dataset into `LLaMA-Factory/data/` (image paths are relative to this directory).

In [ ]:
# Copy dataset files into LLaMA-Factory/data/
!cp /content/vietnamese_ocr/vietnamese_ocr.json /content/LLaMA-Factory/data/
!cp -r /content/vietnamese_ocr/images /content/LLaMA-Factory/data/images
!echo "Images in data/:" $(ls /content/LLaMA-Factory/data/images/*.png | wc -l)
!echo "JSON in data/:" $(ls -lh /content/LLaMA-Factory/data/vietnamese_ocr.json)

In [ ]:
# Verify image path resolution
import json, os

with open("/content/LLaMA-Factory/data/vietnamese_ocr.json", "r") as f:
    data = json.load(f)

sample = data[0]
img_rel = sample["images"][0]  # e.g. "images/txt_00000.png"
img_abs = os.path.join("/content/LLaMA-Factory/data", img_rel)
print(f"Image relative path: {img_rel}")
print(f"Image absolute path: {img_abs}")
print(f"Exists: {os.path.exists(img_abs)}")

# Check all images exist
missing = 0
for item in data:
    for img in item["images"]:
        if not os.path.exists(os.path.join("/content/LLaMA-Factory/data", img)):
            missing += 1
print(f"\nMissing images: {missing}/{len(data)}")

In [ ]:
# Register dataset in dataset_info.json
import json

ds_info_path = "/content/LLaMA-Factory/data/dataset_info.json"
with open(ds_info_path, "r") as f:
    info = json.load(f)

info["vietnamese_ocr"] = {
    "file_name": "vietnamese_ocr.json",
    "formatting": "sharegpt",
    "columns": {
        "messages": "messages",
        "images": "images"
    },
    "tags": {
        "role_tag": "role",
        "content_tag": "content",
        "user_tag": "user",
        "assistant_tag": "assistant"
    }
}

with open(ds_info_path, "w") as f:
    json.dump(info, f, indent=2, ensure_ascii=False)

print("✓ Dataset registered in dataset_info.json")
print(json.dumps(info["vietnamese_ocr"], indent=2))

## 6. Write Training Config (YAML)

Based on `glm_ocr_lora_sft.yaml` from the official guide.

In [ ]:
yaml_content = '''\
### model
model_name_or_path: /content/GLM-OCR
trust_remote_code: true

### method
stage: sft
do_train: true
finetuning_type: lora
lora_rank: 8
lora_target: all

### dataset
dataset: vietnamese_ocr
template: glm_ocr
cutoff_len: 2048
preprocessing_num_workers: 8
dataloader_num_workers: 2

### output
output_dir: /content/glm-ocr-lora-sft
logging_steps: 10
save_steps: 500
plot_loss: true
overwrite_output_dir: true
save_only_model: false
report_to: none

### train
per_device_train_batch_size: 4
gradient_accumulation_steps: 4
learning_rate: 1.0e-4
num_train_epochs: 3
lr_scheduler_type: cosine
warmup_ratio: 0.1
fp16: true
'''

with open("/content/glm_ocr_vn_lora_sft.yaml", "w") as f:
    f.write(yaml_content)

print("✓ Config written to /content/glm_ocr_vn_lora_sft.yaml")
print(yaml_content)

## 7. Launch Training

In [ ]:
import os
os.environ["DISABLE_VERSION_CHECK"] = "1"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [ ]:
!DISABLE_VERSION_CHECK=1 CUDA_VISIBLE_DEVICES=0 \
  llamafactory-cli train /content/glm_ocr_vn_lora_sft.yaml

## 8. Merge LoRA Weights

From the official FAQ.

In [ ]:
!llamafactory-cli export \
  --model_name_or_path /content/GLM-OCR \
  --adapter_name_or_path /content/glm-ocr-lora-sft \
  --template glm_ocr \
  --export_dir /content/glm-ocr-vn-merged \
  --trust_remote_code true

## 9. Save to Google Drive

In [ ]:
!mkdir -p "/content/drive/My Drive/glm-ocr-vn"
!cp -r /content/glm-ocr-vn-merged/* "/content/drive/My Drive/glm-ocr-vn/"
print("✓ Model saved to Drive: My Drive/glm-ocr-vn/")